# Práctica 1 · 02 — Series de la zona, diaria y mensual

**Cambio Climático · Ingeniería Ambiental · ESPAM MFL** · Material CC-IAM-P1-02 · Versión 3.0 · octubre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/P1_02_series.ipynb)

**Objetivo.** Convertir las mallas descargadas en series de tiempo que representen a su zona, con el mismo método para ERA5 y CORe y para las 9 variables.

**Producto.** `serie_mensual_<zona>.csv` y `serie_diaria_<zona>.csv`, con las dos fuentes lado a lado. Son el insumo de todos los análisis siguientes.

**Requisito.** Haber ejecutado `P1_01_descarga_datos` con la misma zona.

**Cómo trabajar.** En Colab: **Archivo › Guardar una copia en Drive**. Escriba solo en la celda **1** y ejecute las demás en orden.

<details><summary>Secuencia de la práctica y variables</summary>


| Orden | Cuaderno | Qué hace | Deja listo |
|---|---|---|---|
| 1 | `P1_01_descarga_datos` | Descarga ERA5 y CORe solo para el recuadro de su zona | `CC_datos/<zona>/` |
| 2 | `P1_02_series` | Recorta con el límite y promedia sobre la zona | Series diaria y mensual |
| 3 | `P1_03_climatologia` | Climatología 1995–2014 de las 8 variables con los dos reanálisis | Tablas y figuras (paso 1) |
| 4 | `P1_04_estacion_vs_reanalisis` | Contrasta una estación con ERA5 y CORe | Métricas y figuras (paso 2) |


| Variable | Columna | Unidad |
|---|---|---|
| Temperatura media, máxima y mínima a 2 m | `tmed`, `tmax`, `tmin` | °C |
| Precipitación | `pr` | mm/día · mm/mes |
| Humedad relativa a 2 m | `hr` | % |
| Humedad específica a 2 m | `q` | g/kg |
| Presión atmosférica en superficie | `ps` | hPa |
| Viento a 10 m: componentes, velocidad y dirección de procedencia | `u10`, `v10`, `ws`, `wd` | m/s · ° |

</details>

© 2026 Cristian Omar Paliz Acosta · ESPAM MFL · Todos los derechos reservados

## 1. Datos del estudiante

In [ ]:
ESTUDIANTE = "Nombre Apellido"
PROVINCIA  = "Manabí"   # la misma en todos los cuadernos
CANTON     = None       # None = toda la provincia; o el nombre de un cantón, p. ej. "Portoviejo"

## 2. Entorno de trabajo

In [ ]:
!pip -q install regionmask

import os, sys, pathlib, subprocess, datetime, platform, json
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import limites, variables as V
from herramientas.reanalisis import era5, core, release
from herramientas.estaciones import calidad

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(".")

import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, geopandas as gpd
Z = limites.zona(PROVINCIA, CANTON)
GEOM, S = Z["geom"], Z["slug"]
DATOS = BASE / "CC_datos" / S            # datos descargados (cuaderno 1)
RES = BASE / "CC_Practica1" / S          # productos de la práctica (cuadernos 2 a 4)
for c in (DATOS, RES): c.mkdir(parents=True, exist_ok=True)
print(f"Zona: {Z['nombre']} ({Z['tipo']}, DPA {Z['codigo']}, región {Z['region']})")
print(f"Datos en {DATOS} · resultados en {RES}")

## 3. Abrir los datos del cuaderno 1

In [ ]:
inv = json.loads((DATOS / "inventario.json").read_text(encoding="utf-8"))
A = {k: (None if v is None else DATOS / v) for k, v in inv["archivos"].items()}
ANIO_INI, ANIO_FIN = inv["periodo"]
E_M = era5.mensual_estandar(A["era5_mensual"])
E_D = xr.open_dataset(A["era5_diario"]) if A.get("era5_diario") else None
C_D = xr.open_dataset(A["core_diario"]) if A.get("core_diario") else None
C_M = xr.open_dataset(A["core_mensual"]) if A.get("core_mensual") else None
for n_, d_ in (("ERA5 mensual", E_M), ("ERA5 diario", E_D), ("CORe diario", C_D), ("CORe mensual", C_M)):
    print(f"{n_:13s}", "—" if d_ is None else f"{dict(d_.sizes)} · {list(d_.data_vars)}")

## 4. La máscara de la zona

Se marcan las celdas de ERA5 cuyo centro cae dentro del límite oficial (si hay menos de cuatro, como en cantones pequeños, las que tocan el límite). El promedio se pondera por el coseno de la latitud.

**CORe** tiene celdas de ~1,9°: se interpola bilinealmente a la malla de ERA5 y se aplica la misma máscara. Eso **no agrega detalle**; la serie sigue representando la escala de CORe.

**Viento:** se promedian las componentes u y v y, con el promedio, se calculan velocidad y dirección (viento medio vectorial). Es menor o igual que el promedio de las velocidades, sobre todo si el viento cambia de dirección en el día.

In [ ]:
MASC, N_CELDAS, METODO = limites.mascara(E_M["tmed"], GEOM)
print(f"ERA5: {N_CELDAS} celdas ({METODO})")
fig, ax = plt.subplots(figsize=(5, 5))
E_M["tmed"].mean("time").where(MASC).plot(ax=ax, cmap="RdYlBu_r", cbar_kwargs={"label": "°C"})
gpd.GeoSeries([GEOM]).boundary.plot(ax=ax, color="k", lw=0.8); ax.set_title(f"Celdas de ERA5 en {Z['nombre']}"); plt.show()

VARS = ["tmed", "tmax", "tmin", "pr", "hr", "q", "ps", "u10", "v10"]
def a_malla_era5(ds): return ds.interp(lat=E_M["lat"], lon=E_M["lon"])
def zona_media(ds, fuente):
    df = pd.DataFrame({v: limites.promedio_area(ds[v], MASC).to_series() for v in VARS if v in ds})
    if {"u10", "v10"} <= set(df): df["ws"], df["wd"] = V.viento(df["u10"], df["v10"])
    return df.add_prefix(f"{fuente}_")

## 5. Serie mensual

ERA5 mensual del Climate Data Store da T media, precipitación, humedad, presión y viento. Tmax y Tmin mensuales de ERA5 se obtienen promediando las diarias.

In [ ]:
partes = []
em = E_M.copy()
if E_D is not None:
    d_m = E_D[["tmax", "tmin"]].resample(time="MS").mean()
    for v in ("tmax", "tmin"): em[v] = d_m[v].reindex_like(em["tmed"])
partes.append(zona_media(em, "ERA5"))
if C_M is not None: partes.append(zona_media(a_malla_era5(C_M), "CORe"))
for p in partes: p.index = p.index.to_period("M").to_timestamp()
MENSUAL = pd.concat(partes, axis=1).round(3); MENSUAL.index.name = "fecha"
display(MENSUAL.describe().round(2).T)

## 6. Serie diaria

Cada fuente define el día de forma distinta: ERA5 de 00 a 24 h hora local; CORe de 01:00 a 01:00; las estaciones del INAMHI suelen medir la lluvia de 07:00 a 07:00. Se declara al comparar valores diarios.

In [ ]:
partes = []
if E_D is not None: partes.append(zona_media(E_D, "ERA5"))
if C_D is not None: partes.append(zona_media(a_malla_era5(C_D), "CORe"))
DIARIA = pd.concat(partes, axis=1).round(3) if partes else None
if DIARIA is not None:
    DIARIA.index.name = "fecha"; display(DIARIA.describe().round(2).T)

## 7. Primera mirada a las series mensuales

In [ ]:
paneles = [("tmed", "T media (°C)"), ("pr", "P (mm/mes)"), ("hr", "HR (%)"), ("q", "q (g/kg)"), ("ps", "Presión (hPa)"), ("ws", "Viento (m/s)")]
fig, ax = plt.subplots(len(paneles), 1, figsize=(12, 13), sharex=True)
for a, (v, et) in zip(ax, paneles):
    for f, c in (("ERA5", "#C2531F"), ("CORe", "#4C78A8")):
        if f"{f}_{v}" in MENSUAL: a.plot(MENSUAL.index, MENSUAL[f"{f}_{v}"], color=c, lw=0.9, label=f)
    a.set_ylabel(et); a.legend(loc="upper right", fontsize=8)
fig.suptitle(f"{Z['nombre']}: series mensuales {ANIO_INI}–{ANIO_FIN}"); fig.tight_layout()
fig.savefig(RES / f"P1_02_series_{S}.png", dpi=200); plt.show()

## 8. Guardar

In [ ]:
import regionmask
MENSUAL.to_csv(RES / f"serie_mensual_{S}.csv", encoding="utf-8")
if DIARIA is not None: DIARIA.to_csv(RES / f"serie_diaria_{S}.csv", encoding="utf-8")
meta = {"material": "CC-IAM-P1-02 v3.0", "estudiante": ESTUDIANTE, "zona": {k: v for k, v in Z.items() if k != "geom"},
        "periodo": [ANIO_INI, ANIO_FIN], "mascara": {"celdas_era5": N_CELDAS, "metodo": METODO,
        "core": "interpolación bilineal a la malla de ERA5 y misma máscara"},
        "viento": "media vectorial de u10 y v10; ws y wd calculados del vector medio",
        "dia": {"ERA5": "00–24 hora local", "CORe": "01:00–01:00 hora local"},
        "columnas_mensual": list(MENSUAL.columns), "columnas_diaria": None if DIARIA is None else list(DIARIA.columns),
        "versiones": {"python": platform.python_version(), "xarray": xr.__version__, "regionmask": regionmask.__version__}}
(RES / f"series_metadatos_{S}.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(sorted(p.name for p in RES.glob("serie*")))

## 9. Preguntas

1. ¿Cuántas celdas de ERA5 representan a su zona? ¿Qué parte del territorio queda mejor o peor representada?
2. ¿En qué variable difieren más ERA5 y CORe? ¿La diferencia es constante o aparece en ciertos años?

**Siguiente:** `P1_03_climatologia`.